In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns



metrics_df = pd.read_csv("../data/12k_all_metrics.csv")
results_df = pd.read_csv("../data/12k_all_results.csv")

all_df = metrics_df.merge(results_df, on="global_id")

In [ ]:
# remove seqs with:
# >= 0.65 GC and <1.85 DNA shannon entropy
# >= 8 EE repeats
# >= 40 AA repeats in DNA
# >= 0.5 KE alpha helix
# then compute the % of seqs that are "Up" in the fourclass; along with how many of the "Up" are in what we filtered out


removed_df = all_df[
    (all_df["dna_gc_content"] >= 0.6) & (all_df["dna_sequence_entropy"] < 1.9) |
    (all_df["num_A_repeats_dna"] >= 45) |
    (all_df["num_E_repeats_aa"] >= 8) |
    (all_df["dssp_ke_alpha_ratio"] >= 0.3) |
    (all_df["num_cysteines"] >=1)
]

# removed_df = all_df[
#     (all_df["boltz_rosetta_A_BC_iptm"] >= 0.5)
# ]

filtered_df = all_df.drop(removed_df.index)


total_seqs = len(all_df)
filtered_seqs = len(filtered_df)
up_seqs = len(all_df[all_df["fourclass"] == "Up"])
recovered_seqs = len(all_df[all_df["leah_12k_detected"] == True])
filtered_up_seqs = len(filtered_df[filtered_df["fourclass"] == "Up"])
filtered_recovered_seqs = len(filtered_df[filtered_df["leah_12k_detected"] == True])

# compute num up over the total of those left after filtering
# and compute num up over the total of those filtered out
percent_up_filtered = filtered_up_seqs / filtered_seqs * 100
percent_up_filtered_out = (up_seqs - filtered_up_seqs) / (total_seqs - filtered_seqs) * 100
percent_recovered_filtered = filtered_recovered_seqs / filtered_seqs * 100
percent_recovered_filtered_out = (recovered_seqs - filtered_recovered_seqs) / (total_seqs - filtered_seqs) * 100

print(f"Total seqs: {total_seqs}")
print(f"Baseline Recovered %: {recovered_seqs / total_seqs * 100:.2f}%")
print(f"Baseline Up %: {up_seqs / total_seqs * 100:.2f}%")
print()
print(f"Seqs left after filtering: {filtered_seqs}\n")
print(f"Recovered % after filtering: {percent_recovered_filtered:.2f}%")
print(f"Recovered % in removed: {percent_recovered_filtered_out:.2f}%")
print()
print(f"Percent Up after filtering: {percent_up_filtered:.2f}%")
print(f"Percent Up in removed: {percent_up_filtered_out:.2f}%")
print()
# check which df the following global_ids are in:
ids_to_check = [1304, 1506, 1892, 2383, 3109, 3494, 3718, 5300, 5624, 5981]
for global_id in ids_to_check:
    in_filtered = global_id in filtered_df["global_id"].values
    in_removed = global_id in removed_df["global_id"].values
    print(f"Global ID {global_id}: In Filtered: {in_filtered}, In Removed: {in_removed}")

Total seqs: 12000
Baseline Recovered %: 56.78%
Baseline Up %: 5.89%

Seqs left after filtering: 8052

Recovered % after filtering: 57.45%
Recovered % in removed: 55.42%

Percent Up after filtering: 6.21%
Percent Up in removed: 5.24%

Global ID 1304: In Filtered: False, In Removed: True
Global ID 1506: In Filtered: False, In Removed: True
Global ID 1892: In Filtered: True, In Removed: False
Global ID 2383: In Filtered: True, In Removed: False
Global ID 3109: In Filtered: False, In Removed: True
Global ID 3494: In Filtered: True, In Removed: False
Global ID 3718: In Filtered: True, In Removed: False
Global ID 5300: In Filtered: True, In Removed: False
Global ID 5624: In Filtered: True, In Removed: False
Global ID 5981: In Filtered: False, In Removed: True


In [ ]:
# Define individual filters and IDs to track
filters = {
    'GC <60%': lambda df: df[df["dna_gc_content"] < 0.6],
    'DNA entropy ≥1.9': lambda df: df[df["dna_sequence_entropy"] >= 1.9],
    'AA repeats <45': lambda df: df[df["num_A_repeats_dna"] < 45],
    'E repeats <8': lambda df: df[df["num_E_repeats_aa"] < 8],
    'K+E helix <30%': lambda df: df[df["dssp_ke_alpha_ratio"] < 0.3],
    'no cysteines': lambda df: df[df["num_cysteines"] == 0],
}

ids_to_check = [1304, 1506, 1892, 2383, 3109, 3494, 3718, 5300, 5624, 5981]

filter_names = list(filters.keys())
filter_funcs = list(filters.values())

# Baseline metrics
baseline_up_pct = (len(all_df[all_df["fourclass"] == "Up"]) / len(all_df)) * 100
baseline_recovered_count = len(all_df[all_df["leah_12k_detected"] == True])
baseline_recovered_pct = (baseline_recovered_count / len(all_df)) * 100

print(f"Baseline: N={len(all_df)}, % Up={baseline_up_pct:.2f}%, % Recovered={baseline_recovered_pct:.2f}%")

In [ ]:
# Sequential Ablation: Apply filters cumulatively
sequential_ablation = []

# Add baseline row
baseline_up_pct = (len(all_df[all_df["fourclass"] == "Up"]) / len(all_df)) * 100
baseline_row = {
    'Filter applied': 'None (baseline)',
    'N remaining': len(all_df),
    '% recovered': baseline_recovered_pct,
    '% ≥2-fold increase': baseline_up_pct,
    'Top-10 removed': None
}
sequential_ablation.append(baseline_row)

# Apply filters sequentially
current_df = all_df.copy()

for i, (filter_name, filter_func) in enumerate(zip(filter_names, filter_funcs)):
    # Apply filter
    current_df = filter_func(current_df)
    
    # Get which ids_to_check have been removed cumulatively (not in current_df)
    cumulative_removed_ids = tuple(id for id in ids_to_check if id not in current_df["global_id"].values)
    
    # Calculate metrics
    n_remaining = len(current_df)
    up_count = len(current_df[current_df["fourclass"] == "Up"])
    up_pct = (up_count / n_remaining * 100) if n_remaining > 0 else 0
    recovered_count = len(current_df[current_df["leah_12k_detected"] == True])
    recovered_pct = (recovered_count / n_remaining * 100) if n_remaining > 0 else 0
    
    row = {
        'Filter applied': f'+ {filter_name}',
        'N remaining': n_remaining,
        '% recovered': recovered_pct,
        '% ≥2-fold increase': up_pct,
        'Top-10 removed': cumulative_removed_ids if cumulative_removed_ids else None
    }
    sequential_ablation.append(row)

# Create DataFrame
sequential_df = pd.DataFrame(sequential_ablation)
print("SEQUENTIAL ABLATION TABLE")
print(sequential_df.to_string(index=False))

**Table S1. Sequential ablation of filtering criteria.** Each row represents the cumulative effect of applying filters sequentially in order. Starting from all 12,000 sequences, each filter removes sequences that fail that criterion. N remaining shows the number of sequences after applying filters up to and including that row. % recovered and % ≥2-fold increase (% Up) show what fraction of remaining sequences meet those criteria. Top-10 removed shows which of the tracked IDs were cumulatively removed by that point in the filtering process.

In [ ]:
# Leave-One-Out Ablation: Apply all filters except one
leave_one_out = []

# Add baseline row (no filters)
leave_one_out.append(baseline_row)

# Apply all filters to get the full filtered set
all_filters_df = all_df.copy()
for filter_func in filter_funcs:
    all_filters_df = filter_func(all_filters_df)

# Get which ids_to_check were removed with all filters
removed_all_filters = tuple(id for id in ids_to_check if id not in all_filters_df["global_id"].values)

# Calculate metrics for all filters applied
n_all_filters = len(all_filters_df)
up_count_all = len(all_filters_df[all_filters_df["fourclass"] == "Up"])
up_pct_all = (up_count_all / n_all_filters * 100) if n_all_filters > 0 else 0
recovered_count_all = len(all_filters_df[all_filters_df["leah_12k_detected"] == True])
recovered_pct_all = (recovered_count_all / n_all_filters * 100) if n_all_filters > 0 else 0

all_filters_row = {
    'Filter applied': 'All filters',
    'N remaining': n_all_filters,
    '% recovered': recovered_pct_all,
    '% ≥2-fold increase': up_pct_all,
    'Top-10 removed': removed_all_filters if removed_all_filters else None
}
leave_one_out.append(all_filters_row)

# Now leave out one filter at a time
for skip_idx in range(len(filter_names)):
    # Apply all filters except skip_idx
    current_df = all_df.copy()
    for i, filter_func in enumerate(filter_funcs):
        if i != skip_idx:
            current_df = filter_func(current_df)
    
    # Get which ids_to_check were removed (not in current_df but in all_df)
    removed_check_ids = tuple(id for id in ids_to_check if id not in current_df["global_id"].values)
    
    # Calculate metrics
    n_remaining = len(current_df)
    up_count = len(current_df[current_df["fourclass"] == "Up"])
    up_pct = (up_count / n_remaining * 100) if n_remaining > 0 else 0
    recovered_count = len(current_df[current_df["leah_12k_detected"] == True])
    recovered_pct = (recovered_count / n_remaining * 100) if n_remaining > 0 else 0
    
    # The filter label shows which filter was LEFT OUT
    skipped_filter = filter_names[skip_idx]
    row = {
        'Filter applied': f'All except {skipped_filter}',
        'N remaining': n_remaining,
        '% recovered': recovered_pct,
        '% ≥2-fold increase': up_pct,
        'Top-10 removed': removed_check_ids if removed_check_ids else None
    }
    leave_one_out.append(row)

# Create DataFrame
leave_one_out_df = pd.DataFrame(leave_one_out)
print("\nLEAVE-ONE-OUT ABLATION TABLE")
print(leave_one_out_df.to_string(index=False))

**Table S2. Leave-one-out ablation of filtering criteria.** Each row shows the result of applying all filters except one, allowing assessment of the individual contribution of each filter. The "All filters" row shows the reference result when all filters are applied. Subsequent rows show what changes when each filter is withheld. N remaining, % recovered, and % ≥2-fold increase (% Up) are computed on the filtered set. Top-10 removed shows which tracked IDs would have been retained if that particular filter were removed.